# FloraScan - Basic CNN
Simple Convolutional Neural Network for plant disease detection.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

print(f"TensorFlow: {tf.__version__}")

## Configuration

In [ ]:
TRAIN_DIR = r'd:\Florascann\dataset_limited\train'
TEST_DIR = r'd:\Florascann\dataset_limited\test'

IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 15

NUM_CLASSES = len(os.listdir(TRAIN_DIR))
print(f"Classes: {NUM_CLASSES}")

## Load Data (No Augmentation)

In [ ]:
# Basic preprocessing - no augmentation
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), 
    batch_size=BATCH_SIZE, class_mode='categorical'
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE), 
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

## Build Basic CNN Model
Simple 3-layer CNN without regularization.

In [ ]:
model = Sequential([
    # Layer 1
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2, 2),
    
    # Layer 2
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    
    # Layer 3
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    
    # Classifier
    Flatten(),
    Dense(256, activation='relu'),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

## Train Model

In [ ]:
print("Training Basic CNN...")
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=test_generator,
    verbose=1
)
print("Training completed!")

## Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], 'b-', label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], 'r-', label='Validation', linewidth=2)
axes[0].set_title('Basic CNN - Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], 'b-', label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'], 'r-', label='Validation', linewidth=2)
axes[1].set_title('Basic CNN - Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(test_generator, verbose=0)
train_acc = history.history['accuracy'][-1]

print("="*50)
print("BASIC CNN RESULTS")
print("="*50)
print(f"Final Training Accuracy: {train_acc*100:.2f}%")
print(f"Final Validation Accuracy: {test_acc*100:.2f}%")
print(f"Overfitting Gap: {(train_acc - test_acc)*100:.2f}%")
print("="*50)
print("\n✅ IMPROVEMENT: CNN captures spatial features much better than ANN!")
print("⚠️ ISSUE: Model is overfitting (high train acc, lower val acc)")
print("   → Need to add regularization (Dropout, BatchNorm)")
print("   → Need data augmentation")

In [ ]:
# Save model
os.makedirs(r'd:\Florascann\models', exist_ok=True)
model.save(r'd:\Florascann\models\model_v1_basic_cnn.h5')
print("Model saved!")